# Partie 2 : Transmission et réception des données

Dans cette partie, vous allez transmettre vos données vers les serveurs applicatifs en utilisant la
technologie LoRaWAN. Vous utiliserez un module LoRa Mote de microchip.  
Documentation du module :
https://ww1.microchip.com/downloads/en/DeviceDoc/RN2483-LoRa-Technology-Module-Command-Reference-User-Guide-DS40001784G.pdf

Ressources externes :
https://github.com/CampusIoT/tutorial/blob/master/rn2483/README.md

## 1. Connexion au module

Afin de pouvoir établir une connexion avec la LoRa Mote, nous allons importer serial.  
Pour installer serial :  
`pip install pyserial`

Il faut que vous trouviez le port tty utilisé par le module.  

Vous pouvez utiliser la commande :
`ls /dev |grep tty`

In [ ]:
#Import libraries
import time
import logging
import serial

In [ ]:
#Configuration
PORT     = '/dev/ttyACM0'
BAUDRATE = 57600

logger=logging.getLogger()
logger.setLevel(logging.DEBUG)


Afin de faciliter la connexion au module, nous vous proposons la fonction ci-dessous.


In [ ]:
def setup_serial(port:str=PORT,baudrate:int=BAUDRATE,bytesize:int=serial.EIGHTBITS,
                        parity:str=serial.PARITY_NONE,stopbits:int=serial.STOPBITS_ONE,
                        dtr:int=False):
    """
    Function to setup my serial connection
    Params:
        port:str        : Port used for my connection, default value PORT
        baudrate:int    : Baudrate, default value BAUDRATE
        bytesize:int    : bytesize, default value serial.EIGHTBITS
        parity:str      : Bit parity, default value serial.PARITY_NONE
        stopbits:int    : Stop bits, default value serial.STOPBITS_ONE
        dtr:bool        : Data Terminal Ready, default value False
    Returns :
        sp:Serial.Serial: A seriaHomel connection
    """
    try:
        sp = serial.Serial()
        sp.port = port
        sp.baudrate = baudrate
        sp.bytesize = bytesize
        sp.parity = parity
        sp.stopbits = stopbits
        sp.dtr=dtr
        sp.open()
        return sp
    except (ValueError,serial.SerialException) as exception:
        logging.critical("Could not open the serial connection.")
        raise exception

In [ ]:
#Test de connexion
module = setup_serial(PORT,BAUDRATE)

CRITICAL:root:Could not open the serial connection.


SerialException: [Errno 2] could not open port /dev/ttyACM0: [Errno 2] No such file or directory: '/dev/ttyACM0'

## 2. Paramétrage du module

Dans cette partie, il est attendu que l'élève recherche les différentes commandes à envoyer au module pour définir les fonctions suivantes.

La fonction send() est fournie.

In [ ]:
def send(sp:serial.Serial,data:str):
    """
    Send data through the serial connection
        Param:
            sp:serial.Serial : serial.Serial object used for the RN2485
            data:str : Data to encode and send
        Returns :
            decoded_response:str: Returns a response if got one
    """
    #Encode data and send it through the serial connection
    data_to_send = (data.rstrip()+"\x0d\x0a").encode()
    sp.write(data_to_send)
    time.sleep(0.2)

    #Wait for a response
    rdata=sp.readline()
    while not rdata:
        rdata = sp.readline()

    #Decode response and send it
    decoded_response = rdata.strip().decode()
    logging.debug("Decoded response : %s",decoded_response)
    return decoded_response

In [ ]:
def reset_module(sp):
    """
    Reset the module
        Param:
        Returns :
            response:str : Response from the module
    """
    #Send the command to reset the module
    command = "sys reset"
    response = send(sp,command)
    return response


In [ ]:
def set_appkey(sp,appkey:str):
    """
    Set the APPKEY
        Param:
            appkey:str
        Returns :
            response:str : Response from the module
    """
    #Send the command to set the appkey
    command = "mac set appkey "+appkey
    response = send(sp,command)
    return response

In [ ]:
def set_joineui(sp,joineui:str):
    """
    Set the JOINEUI
        Param:
            joineui:str
        Returns :
            response:str : Response from the module
    """
    #Send the command to set the joineui
    command = "mac set appeui "+ joineui
    response = send(sp,command)
    return response

In [ ]:
def set_deveui(sp,deveui:str):
    """
    Set the DEVEUI
        Param:
            deveui:str
        Returns :
            response:str : Response from the module
    """
    #Send the command to set the deveui
    command = "mac set deveui "+deveui
    response = send(sp,command)
    return response

### Le data rate, c'est quoi ?  

Le Data Rate (DR) c'est le débit de données. Comme vous l'avez vu en cours et en TP, le facteur d'étalement (Spreading Factor, SF) à un lien direct avec ce dernier.  
Pour pouvoir choisir un facteur d'étalement avec ce module, vous allez devoir choisir la configuration associée au niveau de DR souhaité.  


Extrait de LoRaWAN™ Specification V1.0.2

Data Rate | Spreading Factor | Bandwidth | bits/s
---------|-------------------|----------|--------------------
DR0      | SF12             | 125 kHz   | 250
DR1      | SF11             | 125 kHz   | 440
DR2      | SF10             | 125 kHz   | 980
DR3      | SF9              | 125 kHz   | 1760
DR4      | SF8              | 125 kHz   | 3125
DR5      | SF7              | 125 kHz   | 5470
DR6      | SF7              | 250 kHz   | 11000
DR7      | FSK              | 50 kbps    | 50000

*(Note : Ici, vous utiliserez les configuration de data rate comprises entre 0 et 5)*

#### Rappel - Spreading Factor

Cette vidéo explique la modulation utilisée par LoRa :
https://www.youtube.com/watch?v=dxYY097QNs0

LoRa utilise la modulation CSS (Chirp Spread Spectrum), où les *chirps* (ou symboles) vont transporter les données.  
Le Spreading Factor - ou facteur d'étalement - contrôle l'étalement du chirp dans le temps.  
Plus le facteur d'étalement est elevé, plus le *chirp* est étendu dans le temps; le message reste alors plus longtemps dans l'air (le Time On Air augmente).

Quand le SF est faible, on envoie des messages plus rapidement. On augmente donc le débit au détriment de la portée; en envoyant des signaux plus courts ces derniers sont plus vulnérables aux bruits et interférences.  
Quand le SF est élevé, on envoie des messages plus lentement. Le débit baisse, mais la portée augmente; en envoyant des signaux plus longs ces derniers sont moins vulnérables aux bruits et interférences. Les erreurs sont également plus facilement corrigées grace a la redondance du *chirp*.

In [ ]:
def set_datarate(sp,spreading_factor:int):
    """
    Set the datarate
        Param:
            spreading_factor:int
        Returns :
            response:str : Response from the module
    """
    datarate = abs(spreading_factor -12) #Find the relation between SF and DR
    #Send the command to set the datarate
    command = "mac set dr "+ str(datarate)
    response = send(sp,command)
    return response

In [ ]:
def save_config(sp):
    """
    Save the config
        Param:
            spreading_factor:int
        Returns :
            response:str : Response from the module
    """
    #Send the command to save the current config
    command = "mac save"
    response = send(sp,command)
    return response

Dans cette fonction, vous allez ré-utiliser les fonctions définies plus haut pour configurer le module.


In [ ]:
def config_module(sp,appkey:str,joineui:str,deveui:str,spreading_factor:int):
    """
    Configurate the module
        Param:
            spreading_factor:int
            joineui:str
            deveui:str
            spreading_factor:int
        Returns :
            bool: True if joined, False if not
    """
    logging.info("Resetting device")
    response = reset_module(sp)
    logging.debug("Response : %s",response)

    logging.info("Setting APPKEY : %s",appkey)
    response = set_appkey(sp,appkey)
    logging.debug("Response : %s",response)

    logging.info("Setting JOINEUI : %s",joineui)
    response = set_joineui(sp,joineui)
    logging.debug("Response : %s",response)

    logging.info("Setting DEVEUI : %s",deveui)
    response = set_deveui(sp,deveui)
    logging.debug("Response : %s",response)

    logging.info("Setting the data-rate, spreading_factor = %d",spreading_factor)
    response = set_datarate(sp,spreading_factor)
    logging.debug("Response : %s",response)

    logging.info("Saving mac settings")
    response = save_config(sp)
    logging.debug("Response : %s",response)

    #Here we disable the duty cycle limit on the module to avoid errors
    #This part is given
    for channel in range(0,3):
        #Change duty cycle
        logging.info("Setting channel %s duty cycle to  1.00",channel)
        response = send(sp,f"mac set ch dcycle {channel} 1")
        logging.info("Set %s to dcycle response : %s",channel,response)
        #Channel status to on
        logging.info("Setting channel %s to on",channel)
        response = send(sp,f"mac set ch status {channel} on")
        logging.info("Set %s on response : %s",channel,response)


    #Now we join the network, this part is given
    joining=False
    while not joining:
        logging.info("Preparing to join the network")
        response = send(sp,"mac join otaa")
        logging.info("Mac join otaa response : %s",response)
        if "ok" in response:
            joining=True
        time.sleep(2)

    logging.info("Wating to get the accepted response")
    time.sleep(2) #Wait for accepted response
    ret = sp.readline()
    while not ret:
        ret = sp.readline()
    response = ret.strip().decode()
    logging.info("Status of the join request : %s",response)
    if not "accepted" in response:
        return False
    return True

In [ ]:
#Test de la fonction
response = config_module(module,"0123456789ABCDEF0123456789ABCDEF","DEAD25DEAD25DEAD","DEADDEAD00090009",12)
logging.info("Did we join the network ? %s",response)

NameError: name 'module' is not defined

# 3. Envoi de messages et réception

Dans cette partie, vous allez écrire une fonction permettant d'envoyer un message au LoRa Network Server.

In [ ]:
def send_message(sp,message:hex):
    #Send the command to send the message
    command = "mac tx cnf 5 "+str(message)
    response = send(sp,command)
    return response

send_message(module,0x20)

NameError: name 'module' is not defined

Le LoRa Network Server (ou LNS) est configuré pour publier les messages reçus vers le topic :
`TestTopic/lora/{appid}/{deveui}/{event}`  
A l'aide de mosquitto_sub, vous allez **subscribe** à ce topic depuis une console grâce à la commande suivante :  
`mosquitto_sub -h neocampus.univ-tlse3.fr -t TestTopic/lora/{appid}/{deveui}/# -p 1882 -u test -P test`

Le broker MQTT neOCampus utilise le port 1882. En général, les brokers MQTT utilisent le port 1883. On précise le broker grâce à l'argument `-p` dans la commande ci dessus.
**Si vous essayez d'accéder au broker depuis un réseau externe à l'université, il faudra utiliser le port 10882**

# 4. Application

Utilisez ce que vous avez appris dans ce TP pour réaliser une boucle pour envoyer des messages périodiquement via le module LoRa Mote.  
Attention ! Le time on air (ToA) est une denrée rare en LoRa. Réfléchissez et implémentez des mécanismes pour ne pas le gaspiller (tout en conservant une certaine périodicité dans l’envoi)
Enfin, adaptez votre programme pour envoyer les données du SenseHat

Vous n'êtes pas obligés d'utiliser Jupyter Notebook pour votre programme.

In [ ]:
from sense_hat import SenseHat
from cayennelpp import LppFrame
import time

#Fonction qui retourne les donnees de temperature, d'humidite et de pression encodees au format CayenneLPP
def get_sensor_data():
    sense = SenseHat()
    sense.clear()
    #create frame
    frame = LppFrame()
    # add some sensor data
    #1-temperature
    frame.add_temperature(0, round(sense.get_temperature(),1))
    # 2-Humidity
    frame.add_humidity(6, sense.get_humidity())
    # 3-Pression
    frame.add_pressure(3,sense.get_pressure())
    #creation du buffer
    buffer = bytes(frame)
    return buffer.hex()

#Fonction qui definit la periode de mesure et d'envoie des donnees
def periodic_send_data(sp):
    while(1):
        send_message(sp,get_sensor_data())
        #periode = 5 minutes (300s)
        time.sleep(300)
def start_application():
    #Configuration de module LoRa
    response = config_module(module,"0123456789ABCDEF0123456789ABCDEF","DEAD25DEAD25DEAD","DEADDEAD00090009",12)
    logging.info("Did we join the network ? %s",response)
    #Mesures et Envoi periodiques
    periodic_send_data(module)

start_application()

ModuleNotFoundError: No module named 'sense_hat'